In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, mean, lit, expr
from pyspark.sql.types import DoubleType

In [3]:
spark = SparkSession.builder.appName("Missing and Null Values").getOrCreate()

In [4]:
df = spark.read.csv("/content/dataset - dataset.csv", header=True, inferSchema=True)
print("Original Dataset")
df.show()

Original Dataset
+---+--------+-----+
| id|quantity|price|
+---+--------+-----+
|  1|       3|  150|
|  2|       4|  200|
|  3|       5|  250|
|  4|       3|  150|
|  5|      67| 3350|
|  6|      23| 1150|
|  7|    NULL| NULL|
|  8|      23| 1150|
|  9|    NULL| NULL|
| 10|      12|  600|
| 11|    NULL| NULL|
| 12|      45| 2250|
| 13|       2|  100|
| 14|      23| 1150|
| 15|      56| 2800|
+---+--------+-----+



In [5]:
# Check Null Values Count in Each Column
from pyspark.sql.functions import isnan, count

df.select([
    count(
        when(
            col(c).isNull() |
            (isnan(col(c)) if dict(df.dtypes)[c] in ['double', 'float', 'int'] else False), c)
    ).alias(c)
    for c in df.columns
]).show()

+---+--------+-----+
| id|quantity|price|
+---+--------+-----+
|  0|       3|    3|
+---+--------+-----+



In [6]:
# Method 1: Drop Rows Containing Null Values
df_drop = df.dropna()
print("Dataset After Dropping Null Values")
df_drop.show()

Dataset After Dropping Null Values
+---+--------+-----+
| id|quantity|price|
+---+--------+-----+
|  1|       3|  150|
|  2|       4|  200|
|  3|       5|  250|
|  4|       3|  150|
|  5|      67| 3350|
|  6|      23| 1150|
|  8|      23| 1150|
| 10|      12|  600|
| 12|      45| 2250|
| 13|       2|  100|
| 14|      23| 1150|
| 15|      56| 2800|
+---+--------+-----+



In [8]:
# Method 2: Fill Null Values with Constant Values

df_fill = df.fillna({ "Quantity": 10 })
print("Dataset After Filling Null Values")
df_fill.show()

Dataset After Filling Null Values
+---+--------+-----+
| id|quantity|price|
+---+--------+-----+
|  1|       3|  150|
|  2|       4|  200|
|  3|       5|  250|
|  4|       3|  150|
|  5|      67| 3350|
|  6|      23| 1150|
|  7|      10| NULL|
|  8|      23| 1150|
|  9|      10| NULL|
| 10|      12|  600|
| 11|      10| NULL|
| 12|      45| 2250|
| 13|       2|  100|
| 14|      23| 1150|
| 15|      56| 2800|
+---+--------+-----+



In [12]:
# Method 3: Fill Numeric Null Values with Mean
# First, cast 'Sales' to DoubleType, coercing malformed values to null using try_cast
df_numeric_sales = df.withColumn("price", expr("try_cast(Price as DOUBLE)"))
sales_mean = df_numeric_sales.select(mean(col("price"))).collect()[0][0]
df_mean = df.fillna({  "Price": sales_mean })
print("Dataset After Replacing Null Sales with Mean")
df_mean.show()

Dataset After Replacing Null Sales with Mean
+---+--------+-----+
| id|quantity|price|
+---+--------+-----+
|  1|       3|  150|
|  2|       4|  200|
|  3|       5|  250|
|  4|       3|  150|
|  5|      67| 3350|
|  6|      23| 1150|
|  7|    NULL| 1108|
|  8|      23| 1150|
|  9|    NULL| 1108|
| 10|      12|  600|
| 11|    NULL| 1108|
| 12|      45| 2250|
| 13|       2|  100|
| 14|      23| 1150|
| 15|      56| 2800|
+---+--------+-----+



In [13]:
# Method 4: Replace Specific Missing Values

df_replace = df.na.replace("?", None)
print("Dataset After Replacing '?' with Null")
df_replace.show()
spark.stop()

Dataset After Replacing '?' with Null
+---+--------+-----+
| id|quantity|price|
+---+--------+-----+
|  1|       3|  150|
|  2|       4|  200|
|  3|       5|  250|
|  4|       3|  150|
|  5|      67| 3350|
|  6|      23| 1150|
|  7|    NULL| NULL|
|  8|      23| 1150|
|  9|    NULL| NULL|
| 10|      12|  600|
| 11|    NULL| NULL|
| 12|      45| 2250|
| 13|       2|  100|
| 14|      23| 1150|
| 15|      56| 2800|
+---+--------+-----+

